In [ ]:
import torch
import huggingface_hub
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

validation_fname = huggingface_hub.hf_hub_download(
    repo_id="ArthurConmy/redwood_attn_2l",
    filename="validation_data.pt"
)

data = torch.load(validation_fname, map_location="cpu").long()

print("Type:", type(data))
print("Shape:", data.shape)
print("Dtype:", data.dtype)

print("First 5 sequences:")
print(data[:5])

print("\nFirst sequence:")
print(data[0])

print("Min token id:", data.min().item())
print("Max token id:", data.max().item())

print("Mean token id:", data.float().mean().item())
print("Std token id:", data.float().std().item())

seq_lengths = [len(seq) for seq in data]

print("Num sequences:", len(seq_lengths))
print("Min length:", min(seq_lengths))
print("Max length:", max(seq_lengths))
print("Mean length:", np.mean(seq_lengths))

plt.hist(seq_lengths, bins=50)
plt.title("Sequence length distribution")
plt.xlabel("Length")
plt.ylabel("Count")
plt.show()

flat = data.flatten().tolist()
counter = Counter(flat)

most_common = counter.most_common(20)
tokens, counts = zip(*most_common)

plt.bar(tokens, counts)
plt.title("Top 20 most frequent tokens")
plt.xlabel("Token id")
plt.ylabel("Frequency")
plt.show()

for i in range(3):
    print(f"\nSequence {i}:")
    print(data[i].tolist())


unique_tokens = len(counter)
print("Unique tokens:", unique_tokens)

zero_ratio = (data == 0).float().mean().item()
print("Ratio of token 0:", zero_ratio)


def show_windows(seq, window=10):
    seq = seq.tolist()
    for i in range(0, len(seq) - window, window):
        print(seq[i:i+window])

print("Example sliding windows:")
show_windows(data[0], window=10)


def get_batch(data, batch_size=8):
    idx = torch.randint(0, len(data), (batch_size,))
    return data[idx]

batch = get_batch(data)
print(batch.shape)
print(batch)

ranks = np.arange(len(counter))
freqs = np.array(sorted(counter.values(), reverse=True))

plt.loglog(ranks[:1000], freqs[:1000])
plt.title("Token frequency (Zipf-like distribution)")
plt.xlabel("Rank")
plt.ylabel("Frequency")
plt.show()

print("===== DATASET SUMMARY =====")
print("Sequences:", len(data))
print("Sequence shape:", data.shape)
print("Token range:", (data.min().item(), data.max().item()))
print("Unique tokens:", unique_tokens)
print("Avg sequence length:", np.mean(seq_lengths))